In [ ]:
# Imports

import os
import glob
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
import open_clip
from sklearn.model_selection import StratifiedKFold

In [4]:
# Device & Seed

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print("device:", device)
# print(torch.cuda.is_available())
# print(torch.cuda.get_device_name(0))

device: cuda


In [ ]:
# Configs

DATA_DIR = "./data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR  = os.path.join(DATA_DIR, "test")
CKPT_DIR  = "./checkpoints/"
os.makedirs(CKPT_DIR, exist_ok=True)

NUM_CLASSES      = 100
BATCH_SIZE       = 64
NUM_WORKERS      = 0
EPOCHS_HEAD      = 20
EPOCHS_PER_STAGE = 13
K_FOLDS          = 5
CKPT_PATH        = os.path.join(CKPT_DIR, "best_clip.pt")

CLIP_MODEL      = "ViT-B-32"
CLIP_PRETRAINED = "openai"

# https://github.com/mlfoundations/open_clip
# 400M image-text pairs - https://arxiv.org/abs/2103.00020

In [6]:
# Load CLIP model

model, _, preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL, pretrained=CLIP_PRETRAINED
)
# https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.to
model = model.to(device)

print(f"Model: {CLIP_MODEL} pretrained on {CLIP_PRETRAINED}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")      

C:\venvs\ml312\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Model: ViT-B-32 pretrained on openai
Total params: 151,277,313


In [7]:
# Dataset classes

class LabeledDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.paths = sorted(
            glob.glob(os.path.join(test_dir, "*.jpg")),
            key=lambda x: int(os.path.splitext(os.path.basename(x))[0])
        )
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, os.path.basename(path)

In [ ]:
# Build full sample list (no split — k-fold handles it)

all_paths, all_labels = [], []
for class_id in range(NUM_CLASSES):
    class_dir = os.path.join(TRAIN_DIR, str(class_id))
    for fname in sorted(os.listdir(class_dir)):
        if fname.endswith(".jpg"):
            all_paths.append(os.path.join(class_dir, fname))
            all_labels.append(class_id)

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
print(f"Total samples: {len(all_paths)}")

In [ ]:
# Test dataloader (train/val loaders are created inside train_fold)

test_ds     = TestDataset(TEST_DIR, transform=None)  # preprocess assigned in train_fold
test_loader = None  # initialized after CLIP model loads below

print(f"Test images: {len(TestDataset(TEST_DIR, transform=None))}")

In [10]:
# Classification head

class CLIPClassifier(nn.Module):
  def __init__(self, clip_model, num_classes=NUM_CLASSES, freeze_backbone=True):
      super().__init__()
      self.clip = clip_model
      self.head = nn.Sequential(
          nn.Dropout(0.3),
          nn.Linear(512, num_classes),
      )
      if freeze_backbone:
          for param in self.clip.parameters():
              param.requires_grad = False

  def forward(self, x):
      features = self.clip.encode_image(x)
      features = features.float()
      return self.head(features)

classifier = CLIPClassifier(model, freeze_backbone=True).to(device)
trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
total     = sum(p.numel() for p in classifier.parameters())
print(f"Trainable: {trainable:,} / {total:,} params")

# CLIP's encode_image(x) vision encoder extracts 512-dim feature vector per image, replacing EfficientNet's model.features
# CLIP internally uses float16 for speed, convert to float32 for stable training
# Linear goes from 512 (CLIP's output size for ViT-B-32) to 100 classes
# 0.3 Dropout instead of 0.4 from EfficientNet, CLIP features are more generalizable, less regularization needed.

Trainable: 51,300 / 151,328,613 params


In [11]:
# Trainable: 51,300 / 151,328,613 params
# Compare that to EfficientNet phase 1 which had 1.69M trainable
# Surprisingly, our head is actually much smaller
# CLIP's 512-dim output is more compact than EfficientNet's 1536

In [12]:
# Loss function and training functions

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_one_epoch(model, loader, optimizer, scheduler=None):
  model.train()
  total_loss, total_correct, n = 0.0, 0, 0
  for imgs, labels in tqdm(loader, leave=False):
      imgs, labels = imgs.to(device), labels.to(device)
      optimizer.zero_grad()
      out = model(imgs)
      loss = criterion(out, labels)
      loss.backward()
      optimizer.step()
      total_loss    += loss.item() * imgs.size(0)
      total_correct += (out.argmax(1) == labels).sum().item()
      n             += imgs.size(0)
  if scheduler:
      scheduler.step()
  return total_loss / n, total_correct / n


@torch.no_grad()
def evaluate(model, loader):
  model.eval()
  total_loss, total_correct, n = 0.0, 0, 0
  for imgs, labels in loader:
      imgs, labels = imgs.to(device), labels.to(device)
      out  = model(imgs)
      loss = criterion(out, labels)
      total_loss    += loss.item() * imgs.size(0)
      total_correct += (out.argmax(1) == labels).sum().item()
      n             += imgs.size(0)
  return total_loss / n, total_correct / n

# Identical to EfficientNet notebook. Runs forward pass, computes loss, backprops.

In [ ]:
# train_fold — runs one complete fold: phase 1 + progressive unfreezing

def reset_classifier():
    """Fresh classifier with frozen backbone for each fold."""
    clf = CLIPClassifier(model, freeze_backbone=True).to(device)
    return clf

def train_fold(fold_idx, train_samples, val_samples):
    set_seed(42 + fold_idx)
    clf = reset_classifier()

    tr_ds = LabeledDataset(train_samples, transform=preprocess)
    vl_ds = LabeledDataset(val_samples,   transform=preprocess)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    # ── Phase 1: head only ──
    opt_head = optim.AdamW(filter(lambda p: p.requires_grad, clf.parameters()), lr=1e-3, weight_decay=1e-4)
    sch_head = optim.lr_scheduler.CosineAnnealingLR(opt_head, T_max=EPOCHS_HEAD)
    p1_best_val, p1_best_path = 0.0, None

    print(f"\n--- Fold {fold_idx+1} | Phase 1 ---")
    for epoch in range(EPOCHS_HEAD):
        tr_loss, tr_acc = train_one_epoch(clf, tr_loader, opt_head, sch_head)
        vl_loss, vl_acc = evaluate(clf, vl_loader)
        if vl_acc > p1_best_val and tr_acc < 0.95:
            p1_best_val  = vl_acc
            p1_best_path = os.path.join(CKPT_DIR, f"fold{fold_idx}_p1_best.pt")
            torch.save({"model_state_dict": clf.state_dict()}, p1_best_path)
        print(f"  [{epoch+1}/{EPOCHS_HEAD}] train {tr_acc:.4f} | val {vl_acc:.4f} | gap {tr_acc-vl_acc:.4f}")

    if p1_best_path:
        clf.load_state_dict(torch.load(p1_best_path, map_location=device)["model_state_dict"])
        print(f"  → Loaded best phase 1: val {p1_best_val:.4f}")

    # ── Progressive unfreezing ──
    unfreeze_schedule = [[11,10,9],[8,7,6],[5,4,3],[2,1,0]]
    best_val = 0.0

    for stage, blocks in enumerate(unfreeze_schedule):
        for block_idx in blocks:
            for param in clf.clip.visual.transformer.resblocks[block_idx].parameters():
                param.requires_grad = True
        if stage == 0:
            for param in clf.clip.visual.ln_post.parameters():
                param.requires_grad = True
            clf.clip.visual.proj.requires_grad = True

        opt_stage = optim.AdamW([
            {"params": [p for p in clf.clip.parameters() if p.requires_grad], "lr": 1e-6},
            {"params": clf.head.parameters(), "lr": 1e-4},
        ], weight_decay=1e-4)
        sch_stage = optim.lr_scheduler.CosineAnnealingLR(opt_stage, T_max=EPOCHS_PER_STAGE)

        print(f"\n--- Fold {fold_idx+1} | Stage {stage+1} blocks {blocks} ---")
        for epoch in range(EPOCHS_PER_STAGE):
            tr_loss, tr_acc = train_one_epoch(clf, tr_loader, opt_stage, sch_stage)
            vl_loss, vl_acc = evaluate(clf, vl_loader)
            if vl_acc > best_val:
                best_val = vl_acc
                torch.save({"model_state_dict": clf.state_dict(), "val_acc": vl_acc},
                           os.path.join(CKPT_DIR, f"fold{fold_idx}_best.pt"))
            print(f"  [{epoch+1}/{EPOCHS_PER_STAGE}] train {tr_acc:.4f} | val {vl_acc:.4f} | gap {tr_acc-vl_acc:.4f}")

    return best_val, clf

print("train_fold defined.")

In [ ]:
# K-Fold Cross Validation

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold_results      = []
best_overall_val  = 0.0
best_fold_idx     = 0

for fold_idx, (tr_idx, vl_idx) in enumerate(skf.split(all_paths, all_labels)):
    print(f"\n{'='*60}\nFOLD {fold_idx+1}/{K_FOLDS}\n{'='*60}")
    tr_samples = list(zip(all_paths[tr_idx], all_labels[tr_idx].tolist()))
    vl_samples = list(zip(all_paths[vl_idx], all_labels[vl_idx].tolist()))
    print(f"Train: {len(tr_samples)}, Val: {len(vl_samples)}")

    best_val, _ = train_fold(fold_idx, tr_samples, vl_samples)
    fold_results.append(best_val)

    if best_val > best_overall_val:
        best_overall_val = best_val
        best_fold_idx    = fold_idx

    print(f"\nFold {fold_idx+1} best val: {best_val:.4f}")

print(f"\n{'='*60}")
print(f"K-Fold Results ({K_FOLDS} folds):")
print(f"  Mean: {np.mean(fold_results):.4f}")
print(f"  Std:  {np.std(fold_results):.4f}")
print(f"  All:  {[f'{v:.4f}' for v in fold_results]}")
print(f"  Best: Fold {best_fold_idx+1} ({best_overall_val:.4f})")
print(f"  → Submit: fold{best_fold_idx}_best.pt")

In [ ]:
# Pick my own
# Check which Epoch/Val acc to pick
# for f in sorted(os.listdir(CKPT_DIR)):
#     print(f)

In [ ]:
# Submission — loads best fold checkpoint

test_ds     = TestDataset(TEST_DIR, transform=preprocess)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

chosen_ckpt = os.path.join(CKPT_DIR, f"fold{best_fold_idx}_best.pt")
checkpoint  = torch.load(chosen_ckpt, map_location=device)

# Need a classifier to load into — use last fold's or create fresh
final_clf = reset_classifier()
# Unfreeze all so load_state_dict works correctly
for param in final_clf.parameters():
    param.requires_grad = True
final_clf.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded: fold{best_fold_idx}_best.pt | val_acc {checkpoint['val_acc']:.4f}")

final_clf.eval()
ids, preds = [], []
with torch.no_grad():
    for imgs, names in tqdm(test_loader):
        imgs = imgs.to(device)
        out  = final_clf(imgs)
        ids.extend(names)
        preds.extend(out.argmax(1).cpu().tolist())

sub = pd.DataFrame({"ID": ids, "Label": preds})
sub.to_csv("submission_kfold.csv", index=False)
print(sub.head(10))
print(f"Saved submission_kfold.csv ({len(sub)} rows)")

In [ ]:
# Temp script to determine blocks in the visual transformer

# for name, param in classifier.clip.named_parameters():
#     print(name)

# CLIP's visual transformer has 12 blocks (visual.transformer.resblocks 0 through 11).
# Going to try to unfreeze 3 blocks at a time, from top to bottom.

In [ ]:
# Create training curve graphs

# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches

# # ── EfficientNet B3 (3060, 8+40 epochs) ──
# eff_train = [0.0012,0.0093,0.0336,0.0695,0.0973,0.1147,0.1333,0.1414,
#            0.1448,0.1692,0.2132,0.1981,0.2202,0.2352,0.2874,0.2839,0.2897,0.3117,
#            0.3221,0.3302,0.3488,0.3627,0.3673,0.3917,0.4276,0.4229,0.4160,0.4137,
#            0.4021,0.4403,0.4264,0.4171,0.4461,0.4415,0.4241,0.4415,0.4832,0.4728,
#            0.4739,0.4623,0.4670,0.4716,0.4589,0.4762,0.4670,0.4705,0.4623,0.4577]
# eff_val   = [0.0000,0.0093,0.0231,0.0324,0.0556,0.0648,0.0694,0.0787,
#            0.0787,0.0972,0.1157,0.1250,0.1296,0.1296,0.1806,0.1806,0.2222,0.1991,
#            0.2315,0.2361,0.2407,0.2454,0.2639,0.2546,0.2639,0.2639,0.2731,0.2685,
#            0.2824,0.2824,0.2778,0.2778,0.3009,0.2963,0.2963,0.3009,0.2963,0.2824,
#            0.2963,0.2917,0.3009,0.3056,0.3056,0.3056,0.3009,0.3009,0.2870,0.2870]

# # ── CLIP original (partial — only reported epochs) ──
# clip_x     = [1,2,3,4,5,6,7,8,  9,10,11,12,13,16,17,18,20,21,22,23,25,29,33,36,43]
# clip_train = [0.0834,0.2839,0.3975,0.4890,0.5678,0.6049,0.6385,0.6512,
#             0.6373,0.7161,0.7381,0.7740,0.7926,0.8413,0.8598,0.8853,
#             0.9282,0.9444,0.9594,0.9722,0.9849,0.9919,0.9930,0.9942,0.9942]
# clip_val   = [0.2407,0.3519,0.4537,0.5046,0.5417,0.5602,0.5741,0.5741,
#             0.5972,0.6111,0.6204,0.6528,0.6713,0.6806,0.7037,0.7083,
#             0.7176,0.7222,0.7407,0.7500,0.7593,0.7778,0.7824,0.7870,0.7870]

# # ── CLIP Progressive unfreezing ──
# prog_train = [0.0834,0.2839,0.3975,0.4890,0.5678,0.6049,0.6385,0.6512,
#             0.6315,0.6802,0.6848,0.7068,0.7196,0.7578,0.7474,0.7439,0.7509,0.7370,
#             0.7451,0.7810,0.7845,0.7914,0.8308,0.8181,0.8389,0.8459,0.8459,0.8424,
#             0.8262,0.8760,0.8864,0.8980,0.9038,0.9154,0.9143,0.9247,0.9305,0.9421,
#             0.9200,0.9467,0.9548,0.9629,0.9699,0.9722,0.9722,0.9803,0.9815,0.9791]
# prog_val   = [0.2407,0.3519,0.4537,0.5046,0.5417,0.5602,0.5741,0.5741,
#             0.5880,0.5926,0.6065,0.6111,0.6111,0.6111,0.6157,0.6204,0.6204,0.6204,
#             0.6389,0.6667,0.6852,0.6806,0.6898,0.6991,0.7037,0.7130,0.7130,0.7130,
#             0.7083,0.7269,0.7315,0.7315,0.7361,0.7361,0.7361,0.7361,0.7361,0.7361,
#             0.7407,0.7685,0.7685,0.7778,0.7870,0.7917,0.7963,0.7963,0.7963,0.7963]

# fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# fig.suptitle("Training Results Comparison", fontsize=14, fontweight="bold")

# # ── Left: Val accuracy comparison ──
# ax = axes[0]
# ax.plot(eff_val,   label="EfficientNet-B3", color="steelblue")
# ax.plot(clip_x,  [v for v in clip_val],   label="CLIP (full unfreeze)", color="darkorange", marker="o",
# markersize=3)
# ax.plot(prog_val,  label="CLIP (progressive)", color="green")
# ax.axvline(8 - 0.5,  color="gray", linestyle="--", alpha=0.5, label="Phase 2 start")
# ax.axhline(0.60, color="red", linestyle=":", alpha=0.7, label="60% passing threshold")
# ax.set_title("Validation Accuracy")
# ax.set_xlabel("Epoch")
# ax.set_ylabel("Accuracy")
# ax.legend(fontsize=8)
# ax.set_ylim(0, 1)

# # ── Right: CLIP progressive train vs val with stage markers ──
# ax = axes[1]
# ax.plot(prog_train, label="Train", color="steelblue")
# ax.plot(prog_val,   label="Val",   color="darkorange")
# for x, label in [(8,"P2 S1"),(18,"S2"),(28,"S3"),(38,"S4")]:
#   ax.axvline(x - 0.5, color="gray", linestyle="--", alpha=0.5)
#   ax.text(x, 0.05, label, fontsize=8, color="gray")
# ax.axhline(0.60, color="red", linestyle=":", alpha=0.7, label="60% threshold")
# ax.set_title("CLIP Progressive — Train vs Val")
# ax.set_xlabel("Epoch")
# ax.set_ylabel("Accuracy")
# ax.legend()
# ax.set_ylim(0, 1)

# plt.tight_layout()
# plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
# plt.show()
# print("Saved training_curves.png")